# NB_lab_test_trends
Builds `<catalog>.gold.lab_test_trends` per STTM `sttm_silver_to_gold.xlsx` (sheet `lab_test_trends`).

* **Grain:** one row per test type (test_name) per calendar month
* **Driving table:** `silver.lab_results` grouped by test_name, year/month of test_date
* **Joins (LEFT, optional):** `silver.calendar` on date_key = yyyyMMdd of month start
* **Window:** `LAG(abnormal_rate_pct)` partitioned by test_name ordered by year, month (month-over-month trend)
* **Filters:** none
* **Load type:** FULL — `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
import re

dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip()
if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: {catalog!r}")

src = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"
print("Target:", target_table)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH monthly AS (
    SELECT
        test_name,
        year(test_date)                       AS year,
        month(test_date)                      AS month,
        MIN(date_trunc('MONTH', test_date))   AS month_start,
        COUNT(*)                              AS total_tests,
        COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')  AS critical_test_count
    FROM {src}.lab_results
    GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
    SELECT
        m.*,
        CAST(ROUND(abnormal_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
        CAST(ROUND(critical_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct
    FROM monthly m
),
with_lag AS (
    SELECT
        r.*,
        LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
    FROM rates r
)
SELECT
    CAST(w.test_name AS STRING)                         AS test_name,
    CAST(w.year AS INT)                                 AS year,
    CAST(w.month AS INT)                                AS month,
    CAST(c.month_name AS STRING)                        AS month_name,
    CAST(c.quarter_name AS STRING)                      AS quarter_name,
    CAST(w.total_tests AS INT)                          AS total_tests,
    CAST(w.abnormal_test_count AS INT)                  AS abnormal_test_count,
    CAST(w.critical_test_count AS INT)                  AS critical_test_count,
    w.abnormal_rate_pct,
    w.critical_rate_pct,
    CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
    CAST(
        CASE WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
             ELSE w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct
        END AS DECIMAL(6,2))                            AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {src}.calendar c
       ON c.date_key = CAST(date_format(w.month_start, 'yyyyMMdd') AS INT)
""")

In [0]:
tgt = spark.table(target_table)
tgt_cnt = tgt.count()
dup_cnt = tgt_cnt - tgt.select("test_name", "year", "month").distinct().count()
src_tests = spark.table(f"{src}.lab_results").count()
tgt_tests = tgt.agg({"total_tests": "sum"}).collect()[0][0]
missing_cal = tgt.filter("month_name IS NULL").count()

print(f"rows: {tgt_cnt} | duplicate grain: {dup_cnt} | tests: {tgt_tests} (source {src_tests}) | rows without calendar match: {missing_cal}")
assert dup_cnt == 0, "Grain violated: duplicate test_name + year + month"
assert tgt_tests == src_tests, "Total tests do not reconcile with silver.lab_results"

display(tgt.orderBy("test_name", "year", "month").limit(10))